# Video Game Sales and Engagement Analysis

## Data Loading and Inital Exploration

In [1]:
# Import Libraries
import ast
import re
import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)
pd.set_option('display.max_columns', 50)

In [2]:
games_raw = pd.read_csv('games.csv')
vgsales_raw = pd.read_csv('vgsales.csv')

print(f"games.csv   -> {games_raw.shape[0]} rows, {games_raw.shape[1]} columns")
print(f"vgsales.csv -> {vgsales_raw.shape[0]} rows, {vgsales_raw.shape[1]} columns")

games.csv   -> 1512 rows, 14 columns
vgsales.csv -> 16598 rows, 11 columns


In [3]:
games_raw.head()

,Unnamed: 0,Title,Release Date,Team,Rating,Times Listed,Number of Reviews,Genres,Summary,Reviews,Plays,Playing,Backlogs,Wishlist
0,0,Elden Ring,"Feb 25, 2022","['Bandai Namco Entertainment', 'FromSoftware']",4.5,3.9K,3.9K,"['Adventure', 'RPG']","Elden Ring is a fantasy, action and open world...","[""The first playthrough of elden ring is one o...",17K,3.8K,4.6K,4.8K
1,1,Hades,"Dec 10, 2019",['Supergiant Games'],4.3,2.9K,2.9K,"['Adventure', 'Brawler', 'Indie', 'RPG']",A rogue-lite hack and slash dungeon crawler in...,['convinced this is a roguelike for people who...,21K,3.2K,6.3K,3.6K
2,2,The Legend of Zelda: Breath of the Wild,"Mar 03, 2017","['Nintendo', 'Nintendo EPD Production Group No...",4.4,4.3K,4.3K,"['Adventure', 'RPG']",The Legend of Zelda: Breath of the Wild is the...,['This game is the game (that is not CS:GO) th...,30K,2.5K,5K,2.6K
3,3,Undertale,"Sep 15, 2015","['tobyfox', '8-4']",4.2,3.5K,3.5K,"['Adventure', 'Indie', 'RPG', 'Turn Based Stra...","A small child falls into the Underground, wher...",['soundtrack is tied for #1 with nier automata...,28K,679,4.9K,1.8K
4,4,Hollow Knight,"Feb 24, 2017",['Team Cherry'],4.4,3K,3K,"['Adventure', 'Indie', 'Platform']",A 2D metroidvania with an emphasis on close co...,"[""this games worldbuilding is incredible, with...",21K,2.4K,8.3K,2.3K


In [4]:
vgsales_raw.head()

,Rank,Name,Platform,Year,Genre,Publisher,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales
0,1,Wii Sports,Wii,2006.0,Sports,Nintendo,41.49,29.02,3.77,8.46,82.74
1,2,Super Mario Bros.,NES,1985.0,Platform,Nintendo,29.08,3.58,6.81,0.77,40.24
2,3,Mario Kart Wii,Wii,2008.0,Racing,Nintendo,15.85,12.88,3.79,3.31,35.82
3,4,Wii Sports Resort,Wii,2009.0,Sports,Nintendo,15.75,11.01,3.28,2.96,33.00
4,5,Pokemon Red/Pokemon Blue,GB,1996.0,Role-Playing,Nintendo,11.27,8.89,10.22,1.00,31.37


In [5]:
print("=== games.csv info ===")
games_raw.info()
print("\n=== vgsales.csv info ===")
vgsales_raw.info()

=== games.csv info ===
<class 'pandas.DataFrame'>
RangeIndex: 1512 entries, 0 to 1511
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Unnamed: 0         1512 non-null   int64  
 1   Title              1512 non-null   str    
 2   Release Date       1512 non-null   str    
 3   Team               1511 non-null   str    
 4   Rating             1499 non-null   float64
 5   Times Listed       1512 non-null   str    
 6   Number of Reviews  1512 non-null   str    
 7   Genres             1512 non-null   str    
 8   Summary            1511 non-null   str    
 9   Reviews            1512 non-null   str    
 10  Plays              1512 non-null   str    
 11  Playing            1512 non-null   str    
 12  Backlogs           1512 non-null   str    
 13  Wishlist           1512 non-null   str    
dtypes: float64(1), int64(1), str(12)
memory usage: 165.5 KB

=== vgsales.csv info ===
<class 'pandas.DataFrame'>

In [6]:
print("Missing values, games.csv:")
print(games_raw.isnull().sum())
print("\nMissing values, vgsales.csv:")
print(vgsales_raw.isnull().sum())

Missing values, games.csv:
Unnamed: 0            0
Title                 0
Release Date          0
Team                  1
Rating               13
Times Listed          0
Number of Reviews     0
Genres                0
Summary               1
Reviews               0
Plays                 0
Playing               0
Backlogs              0
Wishlist              0
dtype: int64

Missing values, vgsales.csv:
Rank              0
Name              0
Platform          0
Year            271
Genre             0
Publisher        58
NA_Sales          0
EU_Sales          0
JP_Sales          0
Other_Sales       0
Global_Sales      0
dtype: int64


***Insights***

## Data Cleaning

### games.csv

In [8]:
games = games_raw.copy()

# --- Parse 'K'-suffixed engagement counts into real numbers ---
def parse_k(val):
    val = str(val).strip()
    if val.endswith('K'):
        try:
            return float(val[:-1]) * 1000
        except ValueError:
            return np.nan
    try:
        return float(val)
    except ValueError:
        return np.nan

count_cols = ['Times Listed', 'Number of Reviews', 'Plays', 'Playing', 'Backlogs', 'Wishlist']
for col in count_cols:
    games[col] = games[col].apply(parse_k)

games[count_cols].describe()

,Times Listed,Number of Reviews,Plays,Playing,Backlogs,Wishlist
count,1512.000000,1512.000000,1512.000000,1512.000000,1512.000000,1512.000000
mean,769.459656,769.459656,6253.578704,267.379630,1452.577381,780.540344
std,687.840871,687.840871,5894.977122,426.453368,1341.971968,801.253431
min,0.000000,0.000000,0.000000,0.000000,1.000000,2.000000
25%,284.000000,284.000000,1800.000000,43.000000,461.750000,212.000000
50%,551.000000,551.000000,4200.000000,112.500000,1000.000000,496.000000
75%,1000.000000,1000.000000,9100.000000,298.000000,2100.000000,1100.000000
max,4300.000000,4300.000000,33000.000000,3800.000000,8300.000000,5400.000000


In [9]:
# --- Parse stringified list columns (Genres, Team) into real lists ---
def parse_list(val):
    try:
        parsed = ast.literal_eval(val)
        return parsed if isinstance(parsed, list) else []
    except (ValueError, SyntaxError):
        return []

games['Genres'] = games['Genres'].apply(parse_list)
games['Team'] = games['Team'].apply(parse_list)

games[['Title', 'Genres', 'Team']].head()

,Title,Genres,Team
0,Elden Ring,"[Adventure, RPG]","[Bandai Namco Entertainment, FromSoftware]"
1,Hades,"[Adventure, Brawler, Indie, RPG]",[Supergiant Games]
2,The Legend of Zelda: Breath of the Wild,"[Adventure, RPG]","[Nintendo, Nintendo EPD Production Group No. 3]"
3,Undertale,"[Adventure, Indie, RPG, Turn Based Strategy]","[tobyfox, 8-4]"
4,Hollow Knight,"[Adventure, Indie, Platform]",[Team Cherry]


In [10]:
# --- Parse Release Date; 'releases on TBD' means unreleased -> NaT, not a parsing failure ---
games['Release Date'] = games['Release Date'].replace('releases on TBD', np.nan)
games['Release Date'] = pd.to_datetime(games['Release Date'], format='%b %d, %Y', errors='coerce')
games['Release Year'] = games['Release Date'].dt.year

print(f"Unreleased / unparseable dates: {games['Release Date'].isnull().sum()}")
print(f"Release year range: {games['Release Year'].min():.0f} - {games['Release Year'].max():.0f}")

Unreleased / unparseable dates: 3
Release year range: 1980 - 2025


In [11]:
# --- Handle missing values ---
# Rating (13 missing): a game's rating being unset is meaningful (too few reviews to rate) ->
# leave as NaN and exclude with dropna() at the point of use, rather than imputing a fabricated score
# Team (1 missing) / Summary (1 missing): leave as-is, not used numerically

# --- Standardize categorical text ---
games['Title'] = games['Title'].str.strip()
games['Genres'] = games['Genres'].apply(lambda lst: [g.strip().title() for g in lst])
games['Team'] = games['Team'].apply(lambda lst: [t.strip() for t in lst])

# --- Remove duplicates ---
dup_count = games.duplicated(subset=['Title']).sum()
print(f"Duplicate titles: {dup_count}")
games = games.drop_duplicates(subset=['Title']).reset_index(drop=True)
games = games.drop(columns=['Unnamed: 0'])

print(f"\nFinal games shape: {games.shape}")
games.head()

Duplicate titles: 413

Final games shape: (1099, 14)


,Title,Release Date,Team,Rating,Times Listed,Number of Reviews,Genres,Summary,Reviews,Plays,Playing,Backlogs,Wishlist,Release Year
0,Elden Ring,2022-02-25,"[Bandai Namco Entertainment, FromSoftware]",4.5,3900.0,3900.0,"[Adventure, Rpg]","Elden Ring is a fantasy, action and open world...","[""The first playthrough of elden ring is one o...",17000.0,3800.0,4600.0,4800.0,2022.0
1,Hades,2019-12-10,[Supergiant Games],4.3,2900.0,2900.0,"[Adventure, Brawler, Indie, Rpg]",A rogue-lite hack and slash dungeon crawler in...,['convinced this is a roguelike for people who...,21000.0,3200.0,6300.0,3600.0,2019.0
2,The Legend of Zelda: Breath of the Wild,2017-03-03,"[Nintendo, Nintendo EPD Production Group No. 3]",4.4,4300.0,4300.0,"[Adventure, Rpg]",The Legend of Zelda: Breath of the Wild is the...,['This game is the game (that is not CS:GO) th...,30000.0,2500.0,5000.0,2600.0,2017.0
3,Undertale,2015-09-15,"[tobyfox, 8-4]",4.2,3500.0,3500.0,"[Adventure, Indie, Rpg, Turn Based Strategy]","A small child falls into the Underground, wher...",['soundtrack is tied for #1 with nier automata...,28000.0,679.0,4900.0,1800.0,2015.0
4,Hollow Knight,2017-02-24,[Team Cherry],4.4,3000.0,3000.0,"[Adventure, Indie, Platform]",A 2D metroidvania with an emphasis on close co...,"[""this games worldbuilding is incredible, with...",21000.0,2400.0,8300.0,2300.0,2017.0
